In [1]:
!pip install -q pymupdf pandas sentence-transformers faiss-cpu transformers accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 87.8 MB/s eta 0:00:00:00:0100:01


In [2]:
CURRICULUM_PDF = "/kaggle/input/datasets/mohammedwetwet/connection-data/Chemistry.pdf"
CAPSTONE_PDF = "/kaggle/input/datasets/mohammedwetwet/connection-data/G11_Capstone_Smart_Food_Packaging_revised.pdf"
GOOGLE_SHEET_CSV = ""

LO_ID = "CH.2.04"
LO_DESCRIPTION = "Demonstrate that water is a good solvent"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
NUMBER_OF_RESULTS = 4
MIN_WORDS = 200
MAX_WORDS = 250


In [3]:
import os
import re
import csv
import fitz
import torch
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM

def read_pdf(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"PDF not found: {path}. Add it in Kaggle and copy its exact path into Cell 2."
        )
    pages = []
    with fitz.open(path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text().strip()
            if text:
                pages.append({
                    "source": Path(path).name,
                    "page": page_number,
                    "text": text
                })
    if not pages:
        raise ValueError(f"No readable text found in {path}. This PDF may be scanned.")
    return pages

curriculum_pages = read_pdf(CURRICULUM_PDF)
capstone_pages = read_pdf(CAPSTONE_PDF)
all_pages = curriculum_pages + capstone_pages

print("Chemistry pages:", len(curriculum_pages))
print("Capstone pages:", len(capstone_pages))
print("GPU available:", torch.cuda.is_available())


Chemistry pages: 8
Capstone pages: 4
GPU available: True


In [4]:
def make_chunks(pages, words_per_chunk=220, overlap=40):
    chunks = []
    for page in pages:
        words = page["text"].split()
        start = 0
        while start < len(words):
            end = min(start + words_per_chunk, len(words))
            chunk_text = " ".join(words[start:end])
            if chunk_text:
                chunks.append({
                    "source": page["source"],
                    "page": page["page"],
                    "text": chunk_text
                })
            if end == len(words):
                break
            start = end - overlap
    return chunks

chunks = make_chunks(all_pages)
print("Text chunks:", len(chunks))

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
texts = [chunk["text"] for chunk in chunks]
vectors = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

vector_store = faiss.IndexFlatIP(vectors.shape[1])
vector_store.add(vectors)

def search_documents(question, number=NUMBER_OF_RESULTS, source_name=None):
    question_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, indexes = vector_store.search(question_vector, min(len(chunks), number * 5))
    results = []
    for score, index in zip(scores[0], indexes[0]):
        if index < 0:
            continue
        item = chunks[int(index)]
        if source_name and item["source"] != source_name:
            continue
        result = dict(item)
        result["score"] = float(score)
        results.append(result)
        if len(results) == number:
            break
    return results

def show_documents(documents):
    if not documents:
        return "No relevant passages were found."
    output = ""
    for i, item in enumerate(documents, start=1):
        output += f"[E{i}] {item['source']}, PDF page {item['page']}\\n{item['text']}\\n\\n"
    return output

print("Embedding model and FAISS index are ready.")


Text chunks: 20


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding model and FAISS index are ready.


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

def ask_model(prompt, max_new_tokens=700):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful knowledge assistant. Answer using the retrieved evidence. "
                "If evidence is missing, say so. Do not invent facts, project materials, methods, "
                "measurements, results, or citations. Label possible connections as inferences."
            )
        },
        {"role": "user", "content": prompt}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("LLM ready:", MODEL_NAME)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM ready: Qwen/Qwen2.5-3B-Instruct


In [6]:
if GOOGLE_SHEET_CSV.strip():
    if not os.path.exists(GOOGLE_SHEET_CSV):
        raise FileNotFoundError("Google Sheets CSV path was set, but the file was not found.")
    sheet = pd.read_csv(GOOGLE_SHEET_CSV)
    sheet_text = sheet.fillna("").astype(str).apply(
        lambda row: " | ".join(row), axis=1
    ).tolist()
    sheet_chunks = [
        {"source": Path(GOOGLE_SHEET_CSV).name, "page": i + 1, "text": row}
        for i, row in enumerate(sheet_text) if row.strip()
    ]
    sheet_vectors = embedding_model.encode(
        [item["text"] for item in sheet_chunks],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    vector_store.add(sheet_vectors)
    chunks.extend(sheet_chunks)
    print("Google Sheets CSV rows added:", len(sheet_chunks))
else:
    print("Google Sheets CSV skipped. Add a CSV export path in Cell 2 if your team requires it.")


Google Sheets CSV skipped. Add a CSV export path in Cell 2 if your team requires it.


In [7]:
chat_history = []

def ask_question(question, remember=True):
    history_text = ""
    if chat_history:
        history_text = "\\n".join(
            f"{item['role']}: {item['text']}" for item in chat_history[-6:]
        )

    search_question = question
    if history_text:
        search_question = history_text + "\\nCurrent question: " + question

    evidence = search_documents(search_question)
    evidence_text = show_documents(evidence)

    prompt = f"""
Conversation history:
{history_text if history_text else "No previous conversation."}

Question:
{question}

Retrieved evidence:
{evidence_text}

Answer clearly and directly using the evidence. Include source filename and PDF page where possible.
If the evidence does not answer the question, say that the information was not found in the provided sources.
"""
    answer = ask_model(prompt)
    if remember:
        chat_history.append({"role": "user", "text": question})
        chat_history.append({"role": "assistant", "text": answer})
    return {"answer": answer, "evidence": evidence}

question = input("Ask a question about the PDFs: ")
result = ask_question(question)
print("\\nANSWER\\n", result["answer"])
print("\\nSOURCES\\n", show_documents(result["evidence"]))


Ask a question about the PDFs:  Demonstrate that water is a good solvent


\nANSWER\n Based on the evidence provided, water is demonstrated to be a good solvent through various concepts and learning outcomes related to its ability to dissolve substances. Specifically, the document mentions "CH.2.02 - Demonstrate that water is a good solvent" and further elaborates on this concept throughout the document, including discussions about water's role in purifying water supplies, its effects on chemical and physical properties, and its importance in maintaining aquatic life. This indicates that water's capacity to dissolve a wide variety of substances makes it an excellent solvent.
\nSOURCES\n [E1] Chemistry.pdf, PDF page 2\nqualitative and qualitative analysis. 8. Use analytically equipment correctly 9. interpret the analytical data and results. Essential Questions: How can you get a pure drinkable water? Textbook and Resource Materials: How can you get a pure drinkable water? Evidence of Learning: 1. Use litmus paper for determination of pH 2. Determine pH by use 

In [8]:
follow_up = input("Ask a follow-up question, or press Enter to skip: ").strip()
if follow_up:
    follow_up_result = ask_question(follow_up)
    print("\\nFOLLOW-UP ANSWER\\n", follow_up_result["answer"])
    print("\\nSOURCES\\n", show_documents(follow_up_result["evidence"]))
print("Conversation turns stored:", len(chat_history) // 2)


Ask a follow-up question, or press Enter to skip:  


Conversation turns stored: 1


In [9]:
lo_question = LO_ID + " " + LO_DESCRIPTION
curriculum_evidence = search_documents(
    lo_question, number=4, source_name=Path(CURRICULUM_PDF).name
)
capstone_evidence = search_documents(
    lo_question + " Smart Food Packaging materials moisture preservation",
    number=5,
    source_name=Path(CAPSTONE_PDF).name
)

curriculum_text = show_documents(curriculum_evidence)
capstone_text = show_documents(capstone_evidence)

connection_prompt = f"""
Selected Chemistry Learning Outcome: {LO_ID} — {LO_DESCRIPTION}

Curriculum evidence:
{curriculum_text}

Capstone evidence:
{capstone_text}

Explain the Learning Outcome, state what the Capstone explicitly documents, and explain a possible connection.
Clearly label inferences. Do not claim the project uses a material or method unless the Capstone evidence says so.
If evidence is missing, state that the direct connection is not established.
"""
connection = ask_model(connection_prompt, max_new_tokens=650)
print(connection)


The learning outcome states that students should "Demonstrate that water is a good solvent." This involves understanding water's ability to dissolve various substances and its importance in many chemical reactions.

The capstone evidence documents a specific project related to this learning outcome. It involves designing and fabricating a portable packaging prototype for fresh food, primarily from natural and/or repurposed materials. The project includes integrating a natural pigment-derived sensing indicator that responds to changes in the food's condition. 

This capstone explicitly documents the learning outcome by requiring students to engineer food packaging that demonstrates a measurable delay in deterioration. This process inherently involves understanding water's properties as a solvent because the freshness of the food depends on maintaining its moisture content and preventing microbial growth, both of which require effective water management within the packaging.

Possible co

In [10]:
def word_count(text):
    return len(re.findall(r"\b[\w]+(?:['’\-][\w]+)*\b", text))

essay_evidence = (
    "CURRICULUM EVIDENCE:\\n" + curriculum_text +
    "\\nCAPSTONE EVIDENCE:\\n" + capstone_text +
    "\\nCONNECTION ANALYSIS:\\n" + connection
)

def essay_prompt(extra=""):
    return f"""
Write a complete academic essay with 3 or 4 paragraphs and 200 to 250 words in the essay body.
Aim for about 225 words. Do not count the title.

Learning Outcome: {LO_ID} — {LO_DESCRIPTION}

Connect the Chemistry Learning Outcome to the Smart Food Packaging Capstone using only the evidence below.
Use citations like [E1] (filename, PDF page N) only when supported by the evidence.
Distinguish facts from scientific inferences. Do not invent project materials, methods, results, or personal actions.
If a direct connection is not established, say so. Write the full essay, not an outline or short summary.

Evidence:
{essay_evidence}

{extra}

Return:
TITLE: [title]
ESSAY:
[complete essay]
"""

def parse_essay(text):
    title_match = re.search(r"(?im)^TITLE:\s*(.+)$", text)
    body_match = re.search(r"(?is)\bESSAY:\s*(.*)$", text)
    title = title_match.group(1).strip() if title_match else "Chemistry and Smart Food Packaging"
    body = body_match.group(1).strip() if body_match else text.strip()
    return title, body

essay_title = ""
essay = ""
for attempt in range(1, 6):
    extra = ""
    if attempt > 1:
        extra = (
            f"The previous attempt had {word_count(essay)} words. Rewrite the whole essay and keep it "
            "between 200 and 250 words. Do not merely add a sentence."
        )
    answer = ask_model(essay_prompt(extra), max_new_tokens=1400)
    essay_title, essay = parse_essay(answer)
    total_words = word_count(essay)
    print(f"Essay attempt {attempt}: {total_words} words")
    if MIN_WORDS <= total_words <= MAX_WORDS:
        break

if not MIN_WORDS <= word_count(essay) <= MAX_WORDS:
    raise ValueError(
        f"Essay length check failed after 5 attempts. Last word count: {word_count(essay)}. Run this cell again."
    )

print("\\nTITLE:", essay_title)
print("WORD COUNT:", word_count(essay))
print("\\n", essay)


Essay attempt 1: 354 words
Essay attempt 2: 221 words
\nTITLE: Demonstrating Water's Importance Through Smart Food Packaging
WORD COUNT: 221
\n TITLE: Demonstrating Water's Importance Through Smart Food Packaging

The learning outcome requires students to "Demonstrate that water is a good solvent," emphasizing the substance's capacity to dissolve numerous substances and its significance in various chemical reactions. This principle is intricately linked to the capstone project outlined in the provided evidence, which centers around designing and fabricating a portable food packaging system.

In the capstone project, students create prototypes made from natural and repurposed materials, aiming to delay physical, chemical, and biological deterioration of fresh foods. These designs necessitate meticulous control over water dynamics within the packaging. Maintaining optimal moisture levels is crucial for preserving food freshness, while preventing excessive moisture can prevent bacterial g

In [11]:
test_questions = [
    "What does the selected Chemistry Learning Outcome mean?",
    "What does the Capstone document say about the project?",
    "What exact material did the team test and what were the measured results?"
]

for test_question in test_questions:
    print("\\nQUESTION:", test_question)
    test_result = ask_question(test_question, remember=False)
    print("ANSWER:", test_result["answer"])
    print("EVIDENCE COUNT:", len(test_result["evidence"]))

print("\\nEvaluation is a manual review: check whether each answer is supported by its retrieved pages.")


\nQUESTION: What does the selected Chemistry Learning Outcome mean?
ANSWER: The selected Chemistry Learning Outcome, CH.2.02, means demonstrating that water is a good solvent. This outcome is discussed on PDF page 2 of Chemistry.pdf, which states "Demonstrate that water is a good solvent". It further elaborates on this concept by mentioning that water is crucial for purifying water supplies, affecting chemical and physical properties, and being essential for maintaining aquatic life.
EVIDENCE COUNT: 4
\nQUESTION: What does the Capstone document say about the project?
ANSWER: The provided evidence does not specifically mention what the Capstone document says about the project. It focuses on learning outcomes, key concepts, and skills related to chemistry topics but does not detail the specifics of the Capstone project itself.
EVIDENCE COUNT: 4
\nQUESTION: What exact material did the team test and what were the measured results?
ANSWER: The information was not found in the provided sourc

In [12]:
print("LEARNING OUTCOME:", LO_ID, "-", LO_DESCRIPTION)
print("ESSAY WORD COUNT:", word_count(essay))
print("WORD COUNT CHECK:", "PASS" if MIN_WORDS <= word_count(essay) <= MAX_WORDS else "FAIL")
print("\\nFINAL ESSAY\\n")
print(essay_title)
print()
print(essay)
print("\\nSOURCES USED")
for item in curriculum_evidence + capstone_evidence:
    print(f"- {item['source']}, PDF page {item['page']}")


LEARNING OUTCOME: CH.2.04 - Demonstrate that water is a good solvent
ESSAY WORD COUNT: 221
WORD COUNT CHECK: PASS
\nFINAL ESSAY\n
Demonstrating Water's Importance Through Smart Food Packaging

TITLE: Demonstrating Water's Importance Through Smart Food Packaging

The learning outcome requires students to "Demonstrate that water is a good solvent," emphasizing the substance's capacity to dissolve numerous substances and its significance in various chemical reactions. This principle is intricately linked to the capstone project outlined in the provided evidence, which centers around designing and fabricating a portable food packaging system.

In the capstone project, students create prototypes made from natural and repurposed materials, aiming to delay physical, chemical, and biological deterioration of fresh foods. These designs necessitate meticulous control over water dynamics within the packaging. Maintaining optimal moisture levels is crucial for preserving food freshness, while prev